In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, Imputer, VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

TARGET  = "final_result"
ID_COLS = ["id_student", "label_binary"]   # bukan fitur: id, dan label turunan dari target

# ---- Parameter (harus sama dengan oulad_etl) ----
HDFS   = "hdfs://namenode:9000/data/oulad"
CUTOFF = None                      # None = versi penuh; isi angka untuk versi _d{CUTOFF}
SUFFIX = "" if CUTOFF is None else f"_d{CUTOFF}"
KEYS   = ["code_module", "code_presentation", "id_student"]

spark = (SparkSession.builder
         .appName("oulad-modeling")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "16")
         .getOrCreate())

# ---- Baca tabel fitur hasil ETL ----
ml_df = (spark.read
         .option("header", True)
         .option("inferSchema", True)
         .csv(f"{HDFS}/processed/features_clean{SUFFIX}"))

print(f"{ml_df.count():,} baris, {len(ml_df.columns)} kolom")
ml_df.printSchema()       # cek tipe data dulu

32,593 baris, 52 kolom
root
 |-- code_module: string (nullable = true)
 |-- code_presentation: string (nullable = true)
 |-- id_student: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- region: string (nullable = true)
 |-- highest_education: string (nullable = true)
 |-- imd_band: string (nullable = true)
 |-- age_band: string (nullable = true)
 |-- num_of_prev_attempts: integer (nullable = true)
 |-- studied_credits: integer (nullable = true)
 |-- disability: string (nullable = true)
 |-- final_result: string (nullable = true)
 |-- module_presentation_length: integer (nullable = true)
 |-- date_registration: integer (nullable = true)
 |-- total_click: integer (nullable = true)
 |-- active_days: integer (nullable = true)
 |-- distinct_sites: integer (nullable = true)
 |-- first_activity_day: integer (nullable = true)
 |-- last_activity_day: integer (nullable = true)
 |-- avg_click_per_active_day: double (nullable = true)
 |-- click_dataplus: integer (nullable = tru

In [2]:
ml_df.groupBy("final_result").count().show()

+------------+-----+
|final_result|count|
+------------+-----+
|        Pass|12361|
| Distinction| 3024|
|   Withdrawn|10156|
|        Fail| 7052|
+------------+-----+



In [3]:
from pyspark.sql import functions as F, Window

# 1) Isi 0 untuk kolom hitungan: null di sini berarti "tidak ada aktivitas",
#    bukan "data hilang", jadi median tidak tepat.
click_cols = [c for c in ml_df.columns if c.startswith("click_")]
count_cols = ["total_click", "active_days", "distinct_sites",
              "n_assessments", "n_late", "n_banked"] + click_cols
base = ml_df.fillna(0, subset=count_cols)

w_mod = Window.partitionBy("code_module", "code_presentation")

ml_fe = (base
    # intensitas aktivitas
    .withColumn("clicks_per_site",
        F.col("total_click") / F.greatest(F.col("distinct_sites"), F.lit(1)))
    .withColumn("active_ratio",
        F.col("active_days") / F.col("module_presentation_length"))
    # rentang dan kepadatan aktivitas
    .withColumn("activity_span",
        F.col("last_activity_day") - F.col("first_activity_day"))
    .withColumn("active_density",
        F.col("active_days") / (F.col("activity_span") + 1))
    # sinyal dropout: seberapa awal mahasiswa berhenti (membantu Fail vs Withdrawn)
    .withColumn("days_inactive_at_end",
        F.col("module_presentation_length") - F.col("last_activity_day"))
    # disiplin tugas
    .withColumn("late_ratio",
        F.col("n_late") / F.greatest(F.col("n_assessments"), F.lit(1)))
    # konsistensi nilai (membantu Pass vs Distinction)
    .withColumn("score_range", F.col("max_score") - F.col("min_score"))
    .withColumn("tma_cma_diff", F.col("avg_score_tma") - F.col("avg_score_cma"))
    .withColumn("score_vs_module",
        F.col("avg_score") - F.avg("avg_score").over(w_mod))
    # proporsi jenis aktivitas
    .withColumn("forum_share",
        F.col("click_forumng") / F.greatest(F.col("total_click"), F.lit(1)))
    .withColumn("content_share",
        F.col("click_oucontent") / F.greatest(F.col("total_click"), F.lit(1)))
    .withColumn("quiz_share",
        F.col("click_quiz") / F.greatest(F.col("total_click"), F.lit(1)))
)

In [4]:
num_types = ("int", "bigint", "smallint", "double", "float")
num_cols = [c for c, t in ml_fe.dtypes
            if t in num_types and c not in ID_COLS + [TARGET]]
cat_cols = [c for c, t in ml_fe.dtypes
            if t == "string" and c not in ID_COLS + [TARGET]]

In [5]:
# Pastikan numerik bertipe double (dibutuhkan Imputer)
df = ml_fe.select(*ID_COLS, TARGET,
                  *[F.col(c).cast("double").alias(c) for c in num_cols],
                  *cat_cols)

In [6]:
label_idx = StringIndexer(inputCol=TARGET, outputCol="label", handleInvalid="skip")

In [7]:
# Kategorikal: index lalu one-hot (null ditangani lewat "keep")
cat_idx = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
           for c in cat_cols]
cat_ohe = OneHotEncoder(inputCols=[f"{c}_idx" for c in cat_cols],
                        outputCols=[f"{c}_ohe" for c in cat_cols])


In [8]:
# Numerik: isi missing dengan median
imputer = Imputer(inputCols=num_cols,
                  outputCols=[f"{c}_imp" for c in num_cols],
                  strategy="median")

In [9]:
assembler = VectorAssembler(
    inputCols=[f"{c}_imp" for c in num_cols] + [f"{c}_ohe" for c in cat_cols],
    outputCol="features",
    handleInvalid="keep")

In [10]:
rf = RandomForestClassifier(featuresCol="features", labelCol="label",
                            numTrees=200, maxDepth=8, seed=42)

pipeline = Pipeline(stages=[label_idx, *cat_idx, cat_ohe, imputer, assembler, rf])

train, test = df.randomSplit([0.8, 0.2], seed=42)
model = pipeline.fit(train)
pred = model.transform(test)

for m in ["accuracy", "f1", "weightedPrecision", "weightedRecall"]:
    print(m, MulticlassClassificationEvaluator(
        labelCol="label", predictionCol="prediction", metricName=m).evaluate(pred))

accuracy 0.7266550797914363
f1 0.6860821835983439
weightedPrecision 0.7074929772816181
weightedRecall 0.7266550797914363


In [11]:
labels = model.stages[0].labels   # urutan nama kelas sesuai indeks label

# Baris = kelas sebenarnya, kolom = kelas prediksi
cm = (pred.groupBy("label")
          .pivot("prediction", [float(i) for i in range(len(labels))])
          .count()
          .na.fill(0)
          .orderBy("label"))
cm.show()

print({i: l for i, l in enumerate(labels)})   # pemetaan angka -> nama kelas

+-----+----+----+---+---+
|label| 0.0| 1.0|2.0|3.0|
+-----+----+----+---+---+
|  0.0|2317|   3| 40| 71|
|  1.0|  37|1777|167|  0|
|  2.0| 363| 645|305|  4|
|  3.0| 398|   0|  2|200|
+-----+----+----+---+---+

{0: 'Pass', 1: 'Withdrawn', 2: 'Fail', 3: 'Distinction'}
